# Meta Learning (MAML + Meta Controller) - FFAI Assistant

Meta-parámetros y controlador jerárquico.

In [ ]:
import tensorflow as tf
from tensorflow import keras
print(f"TensorFlow: {tf.__version__}")

In [ ]:
STATE_SIZE, ACTION_SIZE, NUM_GOALS = 256, 15, 8

# MAML Meta-Network: Meta-parámetros iniciales
maml_meta = keras.Sequential([
    keras.layers.Dense(256, activation='relu', input_shape=(STATE_SIZE,)),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(ACTION_SIZE * 2)  # Mean + LogStd
], name='MAML_Meta')

# Meta Controller: State -> Goal Selection
meta_controller = keras.Sequential([
    keras.layers.Dense(256, activation='relu', input_shape=(STATE_SIZE,)),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(NUM_GOALS, activation='softmax')
], name='Meta_Controller')

# Sub-policies: Goal-specific policies
sub_policies_input = keras.Input(shape=(STATE_SIZE + NUM_GOALS,))
x = keras.layers.Dense(256, activation='relu')(sub_policies_input)
x = keras.layers.Dense(128, activation='relu')(x)
sub_policies_output = keras.layers.Dense(ACTION_SIZE, activation='softmax')(x)
sub_policies = keras.Model(sub_policies_input, sub_policies_output, name='Sub_Policies')

print(f"✓ MAML Meta: {maml_meta.count_params():,} params")
print(f"✓ Meta Controller: {meta_controller.count_params():,} params")
print(f"✓ Sub Policies: {sub_policies.count_params():,} params")

In [ ]:
# Exportar
models = {
    'maml_meta': maml_meta,
    'meta_controller': meta_controller,
    'sub_policies': sub_policies
}

for name, model in models.items():
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    with open(f'{name}.tflite', 'wb') as f:
        f.write(converter.convert())
    print(f"✓ Exportado: {name}.tflite")

In [ ]:
from google.colab import files
for name in models.keys():
    files.download(f'{name}.tflite')